# Exploring GTFS stops in the Ruhr area

This notebook loads the static GTFS timetable data for Germany and finds all public transport stops in the Ruhr area.

**Steps:**
1. Load `stops.txt` and look at its structure
2. Filter the stops by a bounding box around the Ruhr area (Duisburg to Dortmund)
3. Count the stop points, stations and unique stop names

**Data source:** [gtfs.de](https://gtfs.de), licensed under [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/)

**Note:** The data is not part of this repository. Download the local transport feed from gtfs.de and unzip it into `ml/data/`.

In [10]:
import pandas as pd
stops_data = pd.read_csv("data/stops.txt" , dtype={"stop_id": str, "parent_station": str})



To review the first 5 rows of the data, we use the funktion .head() 

In [11]:
stops_data.head() 


,stop_name,parent_station,stop_id,stop_lat,stop_lon,location_type,platform_code
0,'s-Heerenberg Gouden Handen,NaN,497060,51.87225,6.247338,1.0,NaN
1,'s-Heerenberg Gouden Handen,497060,485193,51.87228,6.247406,NaN,NaN
2,'s-Heerenberg Molenpoort,NaN,600492,51.87632,6.247432,1.0,NaN
3,'s-Heerenberg Molenpoort,600492,640911,51.87632,6.247432,NaN,NaN
4,'s-Heerenberg Muziekschool,NaN,388043,51.87479,6.254206,1.0,NaN


the funkton shape, shows us that there is 692368 Stops in Germany, for all lines (Buses, Train and Metro stations, each Station may have more than one Plattform).

In [12]:
stops_data.shape

(692368, 7)

the function stops_data.info() is used to get info about the tables 

In [13]:
stops_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 692368 entries, 0 to 692367
Data columns (total 7 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   stop_name       692368 non-null  str    
 1   parent_station  440311 non-null  str    
 2   stop_id         692368 non-null  str    
 3   stop_lat        692368 non-null  float64
 4   stop_lon        692368 non-null  float64
 5   location_type   250425 non-null  float64
 6   platform_code   134390 non-null  str    
dtypes: float64(3), str(4)
memory usage: 37.0 MB


the funcion .colums gives us the colums of our data Set

In [14]:
stops_data.columns

Index(['stop_name', 'parent_station', 'stop_id', 'stop_lat', 'stop_lon',
       'location_type', 'platform_code'],
      dtype='str')

Creating the ruhr Chunk from the data. 

In [15]:
ruhrchunk = stops_data[(stops_data['stop_lat'] > 51.3) & (stops_data['stop_lat'] < 51.65) & (stops_data['stop_lon'] > 6.6) & (stops_data['stop_lon'] < 7.75)]


Review the first few rows of the ruhr_chunk.

In [16]:
ruhrchunk.head()

,stop_name,parent_station,stop_id,stop_lat,stop_lon,location_type,platform_code
2190,Abzw Katernb. Schl.,NaN,551301,51.491024,7.050832,1.0,NaN
5480,"Afferde, Afferder Weg 55",NaN,374033,51.547398,7.666645,1.0,NaN
5481,"Afferde, Afferder Weg 55",374033,340466,51.547398,7.666645,NaN,-
5482,"Afferde, Am Teich",NaN,367236,51.558887,7.650475,1.0,NaN
5483,"Afferde, Am Teich",367236,483292,51.558887,7.650475,NaN,2


Counting Stations where location_type == 1. 

Total number of rows in the Ruhr area

In [17]:
len(ruhrchunk)

27104

How many stations (1), how many platforms (NaN) and other types

In [ ]:
ruhrchunk.value_counts('location_type')

NameError: name 'ruhr_chunk' is not defined

How many different stop names

In [ ]:
ruhrchunk['stop_name'].nunique()

9160

In [ ]:
ruhrchunk[ruhrchunk['stop_name'].str.contains('Dortmund Universität', case=False)]

,stop_name,parent_station,stop_id,stop_lat,stop_lon,location_type,platform_code
135338,Dortmund Universität,NaN,357094,51.492890,7.416033,1.0,NaN
135339,Dortmund Universität,357094,628250,51.492294,7.417425,NaN,NaN
135340,Dortmund Universität,357094,630047,51.492740,7.417708,NaN,NaN
135341,Dortmund Universität,357094,676429,51.493220,7.415925,NaN,NaN


In [ ]:
ruhrchunk['stop_name'].str.contains('Essen ', case=False)

2190      False
5480      False
5481      False
5482      False
5483      False
          ...  
681143    False
688078     True
688100    False
689140    False
689141    False
Name: stop_name, Length: 27104, dtype: bool

## Result

| | Count |
|---|---|
| All stops in Germany | 692,368 |
| Rows in the Ruhr area | 27,104 |
| Stations in the Ruhr area | 8,813 |
| Different stop names in the Ruhr area | 9,160 |

At first I expected one row per stop, but that's not how GTFS works. A big stop like Dortmund Hbf has one row for the station and another row for every platform. That's why we have 27,104 rows but only 8,813 stations.

The number of names is a bit higher than the number of stations. So some stops don't have a station row at all, just separate entries, for example one for each direction of a bus stop.

The names always include the city at the end, like "Altenessen Bahnhof, Essen (Ruhr)". That could be useful later.

**What's not perfect yet:** the rectangle we used (lat 51.3–51.65, lon 6.6–7.75) is only a rough outline of the Ruhr area. It probably catches some places that don't belong to it and misses cities at the edge like Hamm. Next time we could filter by the city in the stop name or by the transport company (`agency.txt`) instead.